# Lab 5: Build an agent that asks before it acts

To start, save a copy to your Drive (File, then Save a copy in Drive) and pick a GPU runtime (Runtime, then Change runtime type, then T4 GPU). This lab goes with Lecture 16 and Milestone 4, and in it you build a small agent that can move files to the trash and send emails on a pretend computer, then decide which of its actions run on their own and which ask the person first. Nothing here touches your real files or email. The outputs below are from running it on a Mac, so your model may plan a little differently.

## 1. Setup

This installs anything that is missing, picks the GPU and loads Qwen2.5-3B-Instruct (about 6 GB to download). Without a GPU it loads the smaller Qwen2.5-1.5B-Instruct, which runs but makes worse plans.

In [1]:
# Install anything this notebook needs that is not already installed.
# Colab already has most of these, so this cell is usually quick.
import importlib.util, subprocess, sys

needed = {"transformers": "transformers", "accelerate": "accelerate"}  # module name: pip package name
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("Installed now:", missing or "nothing, all were already there")

Installed now: nothing, all were already there


In [2]:
import torch

# Use a GPU if there is one: "cuda" on Colab, "mps" on a recent Mac, otherwise the CPU.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Running on", device)

Running on mps


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# A bigger model plans better. On a CPU we fall back to a smaller one so it fits in memory.
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct" if device != "cpu" else "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16 if device != "cpu" else torch.float32
).to(device)
model.eval()
print("Loaded", MODEL_ID)

Loaded Qwen/Qwen2.5-3B-Instruct


## 2. A pretend computer

The person's files live in a dictionary of file names and their text, with a trash folder (so a moved file can come back) and an outbox (so you can see every email the agent sent). `reset()` puts everything back the way it started.

In [4]:
import copy, json, re
from datetime import datetime

START_FILES = {
    "notes/draft-intro.txt": "Intro ideas: start with the club organizer story? (draft)",
    "notes/draft-methods.txt": "Methods: watched 3 club organizers, 40 minutes each (draft)",
    "notes/final-report.txt": "Final report for milestone 1. Do not delete.",
    "notes/meeting.txt": "Team meeting, Oct 3. Decided: Ana builds the prototype, Ben runs the tests on Friday, Sam writes the report.",
    "notes/todo.txt": "Buy coffee filters. Email Sam about the test schedule.",
}

def reset():
    global files, trash, outbox, log
    files = copy.deepcopy(START_FILES)
    trash = {}    # file name -> text, for files moved to the trash
    outbox = []   # every email the agent sent
    log = []      # every action the agent took, written by our code (not by the model)

def show_computer():
    print("Files:", list(files))
    print("Trash:", list(trash))
    print("Sent emails:", len(outbox))

reset()
show_computer()

Files: ['notes/draft-intro.txt', 'notes/draft-methods.txt', 'notes/final-report.txt', 'notes/meeting.txt', 'notes/todo.txt']
Trash: []
Sent emails: 0


## 3. Tools

A tool is a Python function the model may ask to run, but the model never runs anything itself. It writes a tool call with the function's name and arguments, and our code decides whether to run it, ask the person, or refuse. The model reads each tool's docstring, so the docstring has to say in plain words what the tool does, including whether it can be undone.

In [5]:
def list_files(folder: str):
    """List the files in a folder.

    Args:
        folder: The folder name, for example "notes".
    """
    return sorted(name for name in files if name.startswith(folder.rstrip("/") + "/"))


def read_file(path: str):
    """Read the text of a file.

    Args:
        path: The file name, for example "notes/todo.txt".
    """
    if path not in files:
        raise ValueError(f"there is no file called {path}")
    return files[path]


def move_to_trash(path: str):
    """Move a file to the trash. It can be restored later.

    Args:
        path: The file name, for example "notes/old.txt".
    """
    if path not in files:
        raise ValueError(f"there is no file called {path}")
    trash[path] = files.pop(path)
    return f"moved {path} to the trash"


def send_email(to: str, subject: str, body: str):
    """Send an email. A sent email cannot be taken back.

    Args:
        to: The email address to send to.
        subject: The subject line.
        body: The text of the email.
    """
    outbox.append({"to": to, "subject": subject, "body": body})
    return f"sent an email to {to}"


def empty_trash():
    """Delete everything in the trash for good."""
    count = len(trash)
    trash.clear()
    return f"deleted {count} files for good"


TOOLS = {f.__name__: f for f in [list_files, read_file, move_to_trash, send_email, empty_trash]}

## 4. Ask the model which tools it wants to call

Qwen models were trained to answer with tool calls when they are given tools, as one `<tool_call>` block with a small piece of JSON per call. `propose` also gives the model the person's files so it can see what is there before it decides, and nothing runs yet.

In [6]:
def propose(task, temperature=0):
    """Ask the model for the tool calls it wants to make for this task. Nothing runs yet."""
    listing = "\n".join(f"- {name}: {text}" for name, text in files.items())
    messages = [
        {"role": "system", "content": "You manage a person's files and email. Use the tools to do the task."},
        {"role": "user", "content": f"{task}\n\nMy files right now:\n{listing}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tools=list(TOOLS.values()), add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(device)
    settings = {"do_sample": False} if temperature == 0 else {"do_sample": True, "temperature": temperature}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=500, pad_token_id=tokenizer.eos_token_id, **settings)
    reply = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    if "</tool_call>" in reply:  # drop anything the model wrote after its last tool call
        reply = reply[: reply.rfind("</tool_call>") + len("</tool_call>")]

    calls = []
    for raw in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", reply, re.S):
        try:
            call = json.loads(raw)
            calls.append({"tool": call["name"], "args": call.get("arguments", {})})
        except (json.JSONDecodeError, KeyError):
            print("Could not read this tool call:", raw)
    return calls, reply


calls, reply = propose("Move my draft notes to the trash, then email sam@example.com a list of the files you moved.")
print("What the model wrote:\n")
print(reply)

What the model wrote:

<tool_call>
{"name": "list_files", "arguments": {"folder": "notes"}}
</tool_call>
<tool_call>
{"name": "move_to_trash", "arguments": {"path": "notes/draft-intro.txt"}}
</tool_call>
<tool_call>
{"name": "move_to_trash", "arguments": {"path": "notes/draft-methods.txt"}}
</tool_call>
<tool_call>
{"name": "send_email", "arguments": {"to": "sam@example.com", "subject": "Draft Notes Moved to Trash", "body": "I have moved two draft notes to the trash:\n\n- notes/draft-intro.txt: Intro ideas: start with the club organizer story?\n- notes/draft-methods.txt: Methods: watched 3 club organizers, 40 minutes each\n\nPlease feel free to review them and let me know if you'd like to revise them."}}
</tool_call>


## 5. An agent that never asks

The simplest agent runs every tool call as soon as the model writes it. Here it gets a vague request that a person might really type.

In [7]:
def run_without_asking(task):
    calls, _ = propose(task)
    for call in calls:
        try:
            result = TOOLS[call["tool"]](**call["args"])
        except Exception as error:
            result = f"failed: {error}"
        print(f"{call['tool']}({call['args']}) -> {result}")

reset()
run_without_asking("Clean up my notes folder.")
print()
show_computer()

list_files({'folder': 'notes'}) -> ['notes/draft-intro.txt', 'notes/draft-methods.txt', 'notes/final-report.txt', 'notes/meeting.txt', 'notes/todo.txt']
move_to_trash({'path': 'notes/draft-intro.txt'}) -> moved notes/draft-intro.txt to the trash
move_to_trash({'path': 'notes/draft-methods.txt'}) -> moved notes/draft-methods.txt to the trash
move_to_trash({'path': 'notes/todo.txt'}) -> moved notes/todo.txt to the trash
empty_trash({}) -> deleted 3 files for good

Files: ['notes/final-report.txt', 'notes/meeting.txt']
Trash: []
Sent emails: 0


Was this what you would have meant by "clean up"? In the run above, the agent also moved todo.txt to the trash and then emptied the trash, so three files are gone for good, and the person never had a chance to say no. Large models make these mistakes too, for example in the Replit case from Lecture 16, where the agent deleted a production database during a code freeze.

## 6. A policy for each tool

Before writing the loop, you decide what each tool may do without the person. This notebook uses four levels:

| level | what happens | use it when |
|---|---|---|
| `auto` | runs right away and is written in the log | it only looks, or a mistake costs nothing |
| `plan` | runs only after the person approves the plan it is in | a mistake costs something but can be undone |
| `ask` | asks again right before it runs, showing exactly what it will do | a mistake cannot be undone |
| `never` | is refused, even if the person approved the plan | the agent should not do this at all |

`plan` and `ask` are close to level 5 of Parasuraman, Sheridan and Wickens (the computer "executes that suggestion if the human approves"), and `auto` with a log is close to level 7 (it acts, "then necessarily informs the human"). Maes's email agent also let each person choose how much it could do on its own for each kind of action, and here you do that by picking a level for each tool.

In [8]:
POLICY = {
    "list_files": "auto",      # only looks
    "read_file": "auto",       # only looks
    "move_to_trash": "plan",   # can be undone
    "send_email": "ask",       # cannot be taken back
    "empty_trash": "never",    # deletes for good
}

def describe(call):
    """One plain line about a tool call, for showing to the person."""
    a = call["args"]
    if call["tool"] == "move_to_trash":
        return f"Move {a.get('path')} to the trash (can be undone)"
    if call["tool"] == "send_email":
        return f"Send an email to {a.get('to')} with the subject \"{a.get('subject')}\" (cannot be taken back)"
    if call["tool"] == "empty_trash":
        return "Delete everything in the trash for good"
    return f"{call['tool']}({a})"

## 7. Asking the person

`ask_person` shows a question and waits for an answer. If nobody can type, for example when the notebook runs as a script, it uses a default answer and says so. The outputs below were made this way.

In [9]:
def ask_person(question, default="yes"):
    try:
        answer = input(question + " ").strip().lower()
    except Exception:  # no keyboard here
        print(question, default, "(answered automatically)")
        return default
    return answer or default

## 8. The agent loop

`run_agent` asks the model for its tool calls, shows the plan as plain lines with each step's level, and asks the person (`yes`, `no`, or `skip` with step numbers, for example `skip 2 3`). Then it acts one step at a time, asks again before each `ask` step, and stops to ask if a step fails. The report at the end comes from our own log of what ran. In the Replit case, the agent's own account of what happened was wrong.

In [10]:
def record(call, status, result=""):
    log.append({"time": datetime.now().strftime("%H:%M:%S"), "tool": call["tool"],
                "args": call["args"], "status": status, "result": result})


def run_agent(task):
    print("Task:", task, "\n")

    # 1. Propose
    calls, _ = propose(task)
    calls = [c for c in calls if c["tool"] in TOOLS]
    if not calls:
        print("The model did not propose any actions.")
        return

    # Looking needs no permission, so auto tools run now and are only written in the log.
    plan = []
    for call in calls:
        if POLICY[call["tool"]] == "auto":
            record(call, "ran on its own", TOOLS[call["tool"]](**call["args"]))
        else:
            plan.append(call)
    if not plan:
        print("The model did not propose any changes, only looking:")
        for entry in log:
            print(f"  {entry['tool']}({entry['args']}) -> {entry['result']}")
        return

    # 2. Show the plan
    print("Here is what I plan to do:")
    for i, call in enumerate(plan, start=1):
        level = POLICY[call["tool"]]
        note = {"plan": "", "ask": "  [I will ask again before this step]", "never": "  [not allowed, will be skipped]"}[level]
        print(f"  {i}. {describe(call)}{note}")

    # 3. Ask
    answer = ask_person("\nGo ahead? (yes / no / skip 2 3)")
    if answer.startswith("no"):
        print("Stopped. Nothing was changed.")
        return
    skipped = {int(n) for n in re.findall(r"\d+", answer)} if answer.startswith("skip") else set()

    # 4. Act, one step at a time
    for i, call in enumerate(plan, start=1):
        level = POLICY[call["tool"]]
        if i in skipped:
            record(call, "skipped by the person")
            continue
        if level == "never":
            record(call, "refused by the policy")
            continue
        if level == "ask":
            print(f"\nStep {i}: {describe(call)}")
            for key, value in call["args"].items():
                print(f"   {key}: {value}")
            if not ask_person("Do this step? (yes / no)").startswith("y"):
                record(call, "declined by the person")
                continue
        try:
            record(call, "done", TOOLS[call["tool"]](**call["args"]))
        except Exception as error:
            record(call, "failed", str(error))
            if not ask_person(f"Step {i} failed ({error}). Keep going? (yes / no)", default="no").startswith("y"):
                break

    # 5. Report from the log
    print("\nWhat actually happened:")
    for entry in log:
        print(f"  {entry['time']}  {entry['tool']}  {entry['status']}  {entry['result']}")

In [11]:
reset()
run_agent("Move my draft notes to the trash, then email sam@example.com a list of the files you moved.")
print()
show_computer()

Task: Move my draft notes to the trash, then email sam@example.com a list of the files you moved. 



Here is what I plan to do:
  1. Move notes/draft-intro.txt to the trash (can be undone)
  2. Move notes/draft-methods.txt to the trash (can be undone)
  3. Send an email to sam@example.com with the subject "Draft Notes Moved to Trash" (cannot be taken back)  [I will ask again before this step]

Go ahead? (yes / no / skip 2 3) yes (answered automatically)

Step 3: Send an email to sam@example.com with the subject "Draft Notes Moved to Trash" (cannot be taken back)
   to: sam@example.com
   subject: Draft Notes Moved to Trash
   body: I have moved two draft notes to the trash:

- notes/draft-intro.txt: Intro ideas: start with the club organizer story?
- notes/draft-methods.txt: Methods: watched 3 club organizers, 40 minutes each

Please feel free to review them and let me know if you'd like to revise them.
Do this step? (yes / no) yes (answered automatically)

What actually happened:
  02:31:41  list_files  ran on its own  ['notes/draft-intro.txt', 'notes/draft-methods.txt', 'notes/final

In Colab you type the answers yourself. You can run the cell again and answer `skip 1`, or `no` at the email step, and check that the log says what happened.

## 9. Undo

Moving to the trash can be undone because the tool kept the file's text. Sending an email cannot, which is why the policy asks again right before it.

In [12]:
def undo_last():
    """Undo the most recent action that can be undone."""
    for entry in reversed(log):
        if entry["status"] == "done" and entry["tool"] == "move_to_trash":
            path = entry["args"]["path"]
            files[path] = trash.pop(path)
            entry["status"] = "undone"
            print("Restored", path)
            return
    print("Nothing left that can be undone.")

undo_last()
show_computer()

Restored notes/draft-methods.txt
Files: ['notes/final-report.txt', 'notes/meeting.txt', 'notes/todo.txt', 'notes/draft-methods.txt']
Trash: ['notes/draft-intro.txt']
Sent emails: 1


## 10. The vague request again

This runs the request from part 5 through the agent that asks, so you see the plan before anything moves.

In [13]:
reset()
run_agent("Clean up my notes folder.")
print()
show_computer()

Task: Clean up my notes folder. 



Here is what I plan to do:
  1. Move notes/draft-intro.txt to the trash (can be undone)
  2. Move notes/draft-methods.txt to the trash (can be undone)
  3. Move notes/todo.txt to the trash (can be undone)
  4. Delete everything in the trash for good  [not allowed, will be skipped]

Go ahead? (yes / no / skip 2 3) yes (answered automatically)

What actually happened:
  02:31:43  list_files  ran on its own  ['notes/draft-intro.txt', 'notes/draft-methods.txt', 'notes/final-report.txt', 'notes/meeting.txt', 'notes/todo.txt']
  02:31:43  move_to_trash  done  moved notes/draft-intro.txt to the trash
  02:31:43  move_to_trash  done  moved notes/draft-methods.txt to the trash
  02:31:43  move_to_trash  done  moved notes/todo.txt to the trash
  02:31:43  empty_trash  refused by the policy  

Files: ['notes/final-report.txt', 'notes/meeting.txt']
Trash: ['notes/draft-intro.txt', 'notes/draft-methods.txt', 'notes/todo.txt']
Sent emails: 0


## Exercises

These help with Milestone 4 (Decide what it does on its own), where your team decides which actions your tool can take without asking and tests it with 3 people.

**Exercise 1.** Change the policy so that `move_to_trash` is `auto`, and run "Clean up my notes folder." again. What would the person need to see afterwards to notice a file that should not have moved, and how quickly could they undo it? Add that to the report at the end of `run_agent`.

In [14]:
# Exercise 1: your code here

**Exercise 2.** Asking too often has a cost too, since people start approving without reading, which Feng and colleagues call "meaningless approvals" (Lecture 16). Count how many times `ask_person` is called over three different tasks. Then change the plan display so the riskiest step is the easiest to notice, for example by listing `ask` steps first or marking the steps that cannot be undone.

In [15]:
# Exercise 2: your code here

**Exercise 3.** Add one tool from your own project (for example, `post_to_class_forum` or `book_room`), write its docstring and pick its level in `POLICY`. Write one sentence on why, saying what a mistake costs and whether it can be undone, then run a task that uses it.

In [16]:
# Exercise 3: your code here

**Exercise 4.** Let the person change a step instead of only saying yes or no. For example, at the email step, you can accept `edit` and then ask for a new subject or body. Which actions in your own tool should people be able to edit before they run?

In [17]:
# Exercise 4: your code here

**Exercise 5 (harder).** Maes's email agent acted on its own only when it was confident enough, and asked otherwise. Compute how likely the model found each tool call it wrote, for example the average probability of its tokens (Lecture 8), and make the agent ask only for steps below a threshold you choose. Does its confidence go down when the request is vague?

In [18]:
# Exercise 5: your code here